# SEI SLM — Kaggle stage-1 QLoRA

Settings: GPU on, Internet on. Attach the Microsoft GUIDE dataset as an input.

In [ ]:
# 1) Clone your GitHub repo (edit URL after you push)
REPO_URL = "https://github.com/deepakbhatia/small-language-model-security.git"
import os
os.chdir("/kaggle/working")
!rm -rf slms
!git clone {REPO_URL} slms
%cd /kaggle/working/slms
!ls

In [ ]:
# 2) Install train deps (Colab/Kaggle already have torch+CUDA)
!pip install -q "transformers>=4.45" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" \
  "datasets>=3.0" "accelerate>=0.34" "pyyaml" "jsonschema" "scikit-learn" "httpx" "tqdm" "sentencepiece"
import os
os.environ["PYTHONPATH"] = "/kaggle/working/slms/src:/kaggle/working/slms"
os.environ["HF_HUB_DISABLE_XET"] = "1"
# Optional: from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# 3) Find GUIDE CSV in Kaggle inputs
from pathlib import Path
cands = list(Path("/kaggle/input").rglob("GUIDE_Train.csv"))
print("GUIDE candidates:", cands)
GUIDE = str(cands[0]) if cands else ""
assert GUIDE, "Attach Microsoft GUIDE dataset (must contain GUIDE_Train.csv)"

In [ ]:
# 4) Compose a T4-friendly subset + val split
!python scripts/compose_all.py \
  --out data/processed/v1 \
  --guide {GUIDE} \
  --guide-limit 10000 \
  --guide-stage 1 \
  --val-ratio 0.2
!python scripts/verify_jsonl.py --input data/processed/v1/train.jsonl

In [ ]:
# 5) T4-safe stage-1 config override
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path("configs/sft_stage1.yaml").read_text())
cfg.update({
    "train_file": "data/processed/v1/train.jsonl",
    "val_file": "data/processed/v1/val.jsonl",
    "max_length": 2048,
    "epochs": 1,
    "batch_size": 1,
    "grad_accum": 16,
    "eval_steps": 100,
    "save_steps": 100,
})
Path("configs/sft_stage1_kaggle.yaml").write_text(yaml.safe_dump(cfg))
print(cfg)

In [ ]:
# 6) Train (downloads Foundation-Sec-8B on first run)
!python scripts/train_sft.py --config configs/sft_stage1_kaggle.yaml
!ls -lah checkpoints/sei-sft-stage1 || true
!ls -lah checkpoints/sei-sft-stage1/adapter || true

When done, download `/kaggle/working/slms/checkpoints/sei-sft-stage1/` from the notebook output panel.